In [ ]:
# Training config and Drive folder — injected at submit time by ColabTrainingAdapter
CONFIG = {{config}}
FOLDER_ID = '{{folder_id}}'
print('Experiment:', CONFIG['experiment_name'])
print('Model     :', CONFIG['model'])
print('Epochs    :', CONFIG['epochs'])
print('Folder ID :', FOLDER_ID)

In [ ]:
# Authenticate with Google and build Drive client
from google.colab import auth
auth.authenticate_user()

from googleapiclient.discovery import build
drive = build('drive', 'v3')

import io
from googleapiclient.http import MediaIoBaseUpload

def update_status(status: str) -> None:
    buf = io.BytesIO(status.encode())
    media = MediaIoBaseUpload(buf, mimetype='text/plain')
    results = drive.files().list(
        q=f"name='status.txt' and '{FOLDER_ID}' in parents and trashed=false",
        fields='files(id)'
    ).execute()
    files = results.get('files', [])
    if files:
        drive.files().update(fileId=files[0]['id'], media_body=media).execute()
    else:
        drive.files().create(
            body={'name': 'status.txt', 'parents': [FOLDER_ID]},
            media_body=media, fields='id'
        ).execute()

update_status('running')
print('Status -> running')

In [ ]:
import subprocess, sys, re, io
from googleapiclient.http import MediaIoBaseDownload

# Detect CUDA version and reinstall matching PyTorch build
smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(smi.stdout[:400])
m = re.search(r'CUDA Version:\s*(\d+)\.(\d+)', smi.stdout)
if m:
    cuda_ver = int(m.group(1)) * 100 + int(m.group(2))
    cu_tag = 'cu124' if cuda_ver >= 1204 else ('cu121' if cuda_ver >= 1201 else 'cu118')
else:
    cu_tag = 'cu121'
print(f'CUDA driver -> installing torch with {cu_tag}')

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '--force-reinstall',
    'torch', '--index-url', f'https://download.pytorch.org/whl/{cu_tag}'
], check=True)

verify = subprocess.run([
    sys.executable, '-c',
    'import torch; print(f"torch {torch.__version__} cuda={torch.version.cuda} '
    'gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")'
], capture_output=True, text=True)
print(verify.stdout.strip() or verify.stderr.strip())

# Download project wheel from Drive and install
results = drive.files().list(
    q=f"name contains '.whl' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id,name)'
).execute()
wheel_meta = results.get('files', [])
if not wheel_meta:
    update_status('failed')
    raise FileNotFoundError('No .whl found in Drive folder — re-run submit to rebuild.')
wheel_meta = wheel_meta[0]
print(f'Downloading {wheel_meta["name"]}')

buf = io.BytesIO()
downloader = MediaIoBaseDownload(buf, drive.files().get_media(fileId=wheel_meta['id']))
done = False
while not done:
    _, done = downloader.next_chunk()
whl_path = f'/tmp/{wheel_meta["name"]}'
open(whl_path, 'wb').write(buf.getvalue())
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', whl_path], check=True)
print(f'Installed {wheel_meta["name"]}')

In [ ]:
from pathlib import Path
import io
from googleapiclient.http import MediaIoBaseDownload

data_dir = Path('data')
data_dir.mkdir(exist_ok=True)

results = drive.files().list(
    q=f"name contains '.jsonl' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id,name,size)'
).execute()

for f in results.get('files', []):
    buf = io.BytesIO()
    downloader = MediaIoBaseDownload(buf, drive.files().get_media(fileId=f['id']))
    done = False
    while not done:
        _, done = downloader.next_chunk()
    data = buf.getvalue()
    (data_dir / f['name']).write_bytes(data)
    print(f'Downloaded {f["name"]} ({len(data) // 1000} KB)')

In [ ]:
import subprocess, sys

cmd = [
    sys.executable, '-m', 'cli.train',
    '--model', CONFIG['model'],
    '--epochs', str(CONFIG['epochs']),
    '--patience', str(CONFIG['patience']),
    '--warmup-ratio', str(CONFIG['warmup_ratio']),
    '--train-data', 'data/train.jsonl',
    '--eval-data', 'data/eval.jsonl',
    '--output-dir', 'models/checkpoints',
]
print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, check=False)
if result.returncode != 0:
    update_status('failed')
    raise RuntimeError(f'Training failed (exit code {result.returncode})')
print('Training complete.')

In [ ]:
import tarfile
from pathlib import Path
from googleapiclient.http import MediaFileUpload

checkpoint_dir = Path('models/checkpoints')
if not checkpoint_dir.exists():
    update_status('failed')
    raise FileNotFoundError(f'Checkpoint not found at {checkpoint_dir}')

archive_path = Path('/tmp/checkpoint.tar.gz')
with tarfile.open(archive_path, 'w:gz') as tf:
    tf.add(checkpoint_dir, arcname='checkpoints')
print(f'Packaged checkpoint ({archive_path.stat().st_size / 1e6:.1f} MB)')

media = MediaFileUpload(str(archive_path), resumable=True)
results = drive.files().list(
    q=f"name='checkpoint.tar.gz' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id)'
).execute()
existing = results.get('files', [])
if existing:
    drive.files().update(fileId=existing[0]['id'], media_body=media).execute()
else:
    drive.files().create(
        body={'name': 'checkpoint.tar.gz', 'parents': [FOLDER_ID]},
        media_body=media, fields='id'
    ).execute()
print('Uploaded checkpoint.tar.gz to Drive')

In [ ]:
update_status('done')
print('Status -> done')